### Data generation

#### gpt 4o

In [ ]:
import pandas as pd
import numpy as np
import openai
import sys
import os
import re

# ==========================================
# CONFIGURATION
# ==========================================
# PLEASE INSERT YOUR ACTUAL API KEY HERE
API_KEY =  "sk-..."
client = openai.OpenAI(api_key=API_KEY)

MODEL_NAME = "gpt-4o"

# --- FILE NAMES ---
INPUT_DATA_FILE = 'Trousers_new.csv'
OUTPUT_FILE     = 'gpt_predictions_random_full_4o.csv'
# ------------------

# ==========================================
# DATA LOADING
# ==========================================
def load_and_prep_data(filepath):
    print(f"[INIT] Loading data from {filepath}...")
    df = pd.read_csv(filepath)
    sales_cols = [c for c in df.columns if 'count' in c]
    df['mean_sales'] = df[sales_cols].mean(axis=1)
    df['std_sales'] = df[sales_cols].std(axis=1)
    return df

# ==========================================
# PROMPT GENERATORS
# ==========================================
def get_features_text(row):
    features = [
        'prod_name', 'product_type_name', 'graphical_appearance_name',
        'colour_group_name', 'department_name', 'index_name',
        'section_name', 'garment_group_name', 'detail_desc'
    ]
    feature_text = ""
    for feat in features:
        if feat in row.index:
            val = row[feat]
            feature_text += f"  - {feat}: {val}\n"
    return feature_text

def create_full_context_prompt(target_row, df):
    # 1. Target Item Task
    target_features = get_features_text(target_row)
    base_target_text = (
        f"Target Item Task:\n"
        f"Please estimate the demand distribution for:\n"
        f"- Product Name: {target_row['prod_name']}\n"
        f"- Features:\n{target_features}"
    )

    # 2. Context: Randomly sample 100 items (excluding target)
    # This runs fresh for EVERY item
    other_items = df[df.index != target_row.name]

    if len(other_items) > 100:
        #other_items = other_items.sample(n=100, random_state=42)
        other_items = other_items.sample(n=100)

    # --- NEW: Calculate Stats for this specific batch of 100 ---
    batch_mean = other_items['mean_sales'].mean()
    batch_std  = other_items['std_sales'].mean()
    # -----------------------------------------------------------

    examples_str = f"Context ({len(other_items)} Reference Items with known Demand Statistics):\n"

    for i, (_, row) in enumerate(other_items.iterrows(), 1):
        examples_str += f"\n[Reference Item {i}]\n"
        examples_str += f"Features:\n{get_features_text(row)}"
        examples_str += f"KNOWN STATISTICS -> Mean: {row['mean_sales']:.2f}, Std: {row['std_sales']:.2f}\n"
        examples_str += "-" * 30

    intro = "You are an expert in inventory demand forecasting. Use the provided reference items to estimate statistics for the Target Item.\n\n"

    full_prompt_text = intro + examples_str + "\n\n" + base_target_text

    return full_prompt_text, batch_mean, batch_std

def create_descriptive_task_prompt():
    return (
        "Task: Estimate the underlying Normal distribution parameters for the Target Item.\n\n"
        "STRICT OUTPUT FORMAT:\n"
        "You must start your response with exactly this line:\n"
        "PREDICTION -> Mean: [Value], Std: [Value]\n\n"
        "Then, on a new line, provide your:\n"
        "REASONING -> [Your detailed explanation here]"
    )

# ==========================================
# API EXECUTION
# ==========================================
def call_gpt_api(prompt):
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "system", "content": "You are a helpful forecasting assistant."},
                      {"role": "user", "content": prompt}],
            #temperature=0.0,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"\n[ERROR] API Call Failed: {e}")
        return None

def extract_metrics(text):
    pred_mean, pred_std = None, None
    match_line = re.search(r"PREDICTION ->(.*?)(?:\n|$)", text, re.IGNORECASE)

    if match_line:
        prediction_text = match_line.group(1)
        mean_match = re.search(r"Mean[:\s=]+([\d\.]+)", prediction_text, re.IGNORECASE)
        std_match = re.search(r"(?:Std|Deviation)[:\s=]+([\d\.]+)", prediction_text, re.IGNORECASE)

        if mean_match:
            try: pred_mean = float(mean_match.group(1))
            except: pass
        if std_match:
            try: pred_std = float(std_match.group(1))
            except: pass

    return pred_mean, pred_std

# ==========================================
# MAIN LOOP
# ==========================================
if __name__ == "__main__":
    if not os.path.exists(INPUT_DATA_FILE):
        print(f"Error: {INPUT_DATA_FILE} not found.")
    else:
        df = load_and_prep_data(INPUT_DATA_FILE)
        results = []

        total_items = len(df)
        print(f"\n[START] Processing ALL {total_items} items (Dynamic Context Strategy)...")

        for idx in range(total_items):
            target_row = df.iloc[idx]
            article_id = target_row['article_id']

            # 1. Generate Prompt AND get the stats for the specific context used
            context_prompt, context_batch_mean, context_batch_std = create_full_context_prompt(target_row, df)

            full_prompt = f"{context_prompt}\n{create_descriptive_task_prompt()}"

            sys.stdout.write(f"\rItem {idx+1}/{total_items} (ID: {article_id})...")
            sys.stdout.flush()

            output = call_gpt_api(full_prompt)

            if output:
                pred_mean, pred_std = extract_metrics(output)

                true_mean = target_row['mean_sales']
                if pred_mean is not None:
                    error_bias = pred_mean - true_mean
                    abs_error = abs(error_bias)
                else:
                    error_bias = None
                    abs_error = None

                results.append({
                    "article_id": article_id,
                    "prod_name": target_row['prod_name'],

                    # Truth
                    "true_mean": true_mean,
                    "true_std": target_row['std_sales'],

                    # Prediction
                    "pred_mean": pred_mean,
                    "pred_std": pred_std,
                    "error_bias": error_bias,
                    "abs_error": abs_error,

                    # --- CONTEXT STATS FOR THIS ROW ---
                    # Since context changes every time, this will vary per row
                    "context_batch_mean": context_batch_mean,
                    "context_batch_std": context_batch_std,

                    "gpt_full_response": output
                })

            # Save incrementally
            if (idx + 1) % 10 == 0:
                pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)

        # FINAL SAVE
        if results:
            pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)
            print(f"\n\n[DONE] Results saved to: {OUTPUT_FILE}")

#### gpt-5-mini

In [ ]:
import pandas as pd
import numpy as np
import openai
import sys
import os
import re

# ==========================================
# CONFIGURATION
# ==========================================
# PLEASE INSERT YOUR ACTUAL API KEY HERE
API_KEY =  "sk-..."
client = openai.OpenAI(api_key=API_KEY)

MODEL_NAME = "gpt-5-mini"

# --- FILE NAMES ---
INPUT_DATA_FILE = 'Trousers_new.csv'
OUTPUT_FILE     = 'gpt_predictions_random_full_5_mini.csv'
# ------------------

# ==========================================
# DATA LOADING
# ==========================================
def load_and_prep_data(filepath):
    print(f"[INIT] Loading data from {filepath}...")
    df = pd.read_csv(filepath)
    sales_cols = [c for c in df.columns if 'count' in c]
    df['mean_sales'] = df[sales_cols].mean(axis=1)
    df['std_sales'] = df[sales_cols].std(axis=1)
    return df

# ==========================================
# PROMPT GENERATORS
# ==========================================
def get_features_text(row):
    features = [
        'prod_name', 'product_type_name', 'graphical_appearance_name',
        'colour_group_name', 'department_name', 'index_name',
        'section_name', 'garment_group_name', 'detail_desc'
    ]
    feature_text = ""
    for feat in features:
        if feat in row.index:
            val = row[feat]
            feature_text += f"  - {feat}: {val}\n"
    return feature_text

def create_full_context_prompt(target_row, df):
    # 1. Target Item Task
    target_features = get_features_text(target_row)
    base_target_text = (
        f"Target Item Task:\n"
        f"Please estimate the demand distribution for:\n"
        f"- Product Name: {target_row['prod_name']}\n"
        f"- Features:\n{target_features}"
    )

    # 2. Context: Randomly sample 100 items (excluding target)
    # This runs fresh for EVERY item
    other_items = df[df.index != target_row.name]

    if len(other_items) > 100:
        #other_items = other_items.sample(n=100, random_state=42)
        other_items = other_items.sample(n=100)

    # --- NEW: Calculate Stats for this specific batch of 100 ---
    batch_mean = other_items['mean_sales'].mean()
    batch_std  = other_items['std_sales'].mean()
    # -----------------------------------------------------------

    examples_str = f"Context ({len(other_items)} Reference Items with known Demand Statistics):\n"

    for i, (_, row) in enumerate(other_items.iterrows(), 1):
        examples_str += f"\n[Reference Item {i}]\n"
        examples_str += f"Features:\n{get_features_text(row)}"
        examples_str += f"KNOWN STATISTICS -> Mean: {row['mean_sales']:.2f}, Std: {row['std_sales']:.2f}\n"
        examples_str += "-" * 30

    intro = "You are an expert in inventory demand forecasting. Use the provided reference items to estimate statistics for the Target Item.\n\n"

    full_prompt_text = intro + examples_str + "\n\n" + base_target_text

    return full_prompt_text, batch_mean, batch_std

def create_descriptive_task_prompt():
    return (
        "Task: Estimate the underlying Normal distribution parameters for the Target Item.\n\n"
        "STRICT OUTPUT FORMAT:\n"
        "You must start your response with exactly this line:\n"
        "PREDICTION -> Mean: [Value], Std: [Value]\n\n"
        "Then, on a new line, provide your:\n"
        "REASONING -> [Your detailed explanation here]"
    )

# ==========================================
# API EXECUTION
# ==========================================
def call_gpt_api(prompt):
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "system", "content": "You are a helpful forecasting assistant."},
                      {"role": "user", "content": prompt}],
            #temperature=0.0,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"\n[ERROR] API Call Failed: {e}")
        return None

def extract_metrics(text):
    pred_mean, pred_std = None, None
    match_line = re.search(r"PREDICTION ->(.*?)(?:\n|$)", text, re.IGNORECASE)

    if match_line:
        prediction_text = match_line.group(1)
        mean_match = re.search(r"Mean[:\s=]+([\d\.]+)", prediction_text, re.IGNORECASE)
        std_match = re.search(r"(?:Std|Deviation)[:\s=]+([\d\.]+)", prediction_text, re.IGNORECASE)

        if mean_match:
            try: pred_mean = float(mean_match.group(1))
            except: pass
        if std_match:
            try: pred_std = float(std_match.group(1))
            except: pass

    return pred_mean, pred_std

# ==========================================
# MAIN LOOP
# ==========================================
if __name__ == "__main__":
    if not os.path.exists(INPUT_DATA_FILE):
        print(f"Error: {INPUT_DATA_FILE} not found.")
    else:
        df = load_and_prep_data(INPUT_DATA_FILE)
        results = []

        total_items = len(df)
        print(f"\n[START] Processing ALL {total_items} items (Dynamic Context Strategy)...")

        for idx in range(total_items):
            target_row = df.iloc[idx]
            article_id = target_row['article_id']

            # 1. Generate Prompt AND get the stats for the specific context used
            context_prompt, context_batch_mean, context_batch_std = create_full_context_prompt(target_row, df)

            full_prompt = f"{context_prompt}\n{create_descriptive_task_prompt()}"

            sys.stdout.write(f"\rItem {idx+1}/{total_items} (ID: {article_id})...")
            sys.stdout.flush()

            output = call_gpt_api(full_prompt)

            if output:
                pred_mean, pred_std = extract_metrics(output)

                true_mean = target_row['mean_sales']
                if pred_mean is not None:
                    error_bias = pred_mean - true_mean
                    abs_error = abs(error_bias)
                else:
                    error_bias = None
                    abs_error = None

                results.append({
                    "article_id": article_id,
                    "prod_name": target_row['prod_name'],

                    # Truth
                    "true_mean": true_mean,
                    "true_std": target_row['std_sales'],

                    # Prediction
                    "pred_mean": pred_mean,
                    "pred_std": pred_std,
                    "error_bias": error_bias,
                    "abs_error": abs_error,

                    # --- CONTEXT STATS FOR THIS ROW ---
                    # Since context changes every time, this will vary per row
                    "context_batch_mean": context_batch_mean,
                    "context_batch_std": context_batch_std,

                    "gpt_full_response": output
                })

            # Save incrementally
            if (idx + 1) % 10 == 0:
                pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)

        # FINAL SAVE
        if results:
            pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)
            print(f"\n\n[DONE] Results saved to: {OUTPUT_FILE}")

#### gemeni

In [ ]:
import pandas as pd
import numpy as np
from google import genai
from google.genai import types
import sys
import os
import re
import time

# ==========================================
# CONFIGURATION
# ==========================================
# PLEASE INSERT YOUR ACTUAL GOOGLE AI STUDIO API KEY HERE
API_KEY = "..."

# Initialize the Client (New SDK Structure)
client = genai.Client(api_key=API_KEY)

# Use the latest available Flash model
MODEL_NAME = "gemini-2.5-flash"

# --- FILE NAMES ---
INPUT_DATA_FILE = 'Trousers_new.csv'
OUTPUT_FILE     = 'gemini_predictions_random_full_flash_new.csv'
# ------------------

# ==========================================
# DATA LOADING
# ==========================================
def load_and_prep_data(filepath):
    print(f"[INIT] Loading data from {filepath}...")
    df = pd.read_csv(filepath)
    sales_cols = [c for c in df.columns if 'count' in c]
    df['mean_sales'] = df[sales_cols].mean(axis=1)
    df['std_sales'] = df[sales_cols].std(axis=1)
    return df

# ==========================================
# PROMPT GENERATORS
# ==========================================
def get_features_text(row):
    features = [
        'prod_name', 'product_type_name', 'graphical_appearance_name',
        'colour_group_name', 'department_name', 'index_name',
        'section_name', 'garment_group_name', 'detail_desc'
    ]
    feature_text = ""
    for feat in features:
        if feat in row.index:
            val = row[feat]
            feature_text += f"  - {feat}: {val}\n"
    return feature_text

def create_full_context_prompt(target_row, df):
    # 1. Target Item Task
    target_features = get_features_text(target_row)
    base_target_text = (
        f"Target Item Task:\n"
        f"Please estimate the demand distribution for:\n"
        f"- Product Name: {target_row['prod_name']}\n"
        f"- Features:\n{target_features}"
    )

    # 2. Context: Randomly sample 100 items (excluding target)
    other_items = df[df.index != target_row.name]

    if len(other_items) > 100:
        other_items = other_items.sample(n=100)

    # --- Calculate Stats for this specific batch ---
    batch_mean = other_items['mean_sales'].mean()
    batch_std  = other_items['std_sales'].mean()
    # -----------------------------------------------

    examples_str = f"Context ({len(other_items)} Reference Items with known Demand Statistics):\n"

    for i, (_, row) in enumerate(other_items.iterrows(), 1):
        examples_str += f"\n[Reference Item {i}]\n"
        examples_str += f"Features:\n{get_features_text(row)}"
        examples_str += f"KNOWN STATISTICS -> Mean: {row['mean_sales']:.2f}, Std: {row['std_sales']:.2f}\n"
        examples_str += "-" * 30

    intro = "You are an expert in inventory demand forecasting. Use the provided reference items to estimate statistics for the Target Item.\n\n"

    full_prompt_text = intro + examples_str + "\n\n" + base_target_text

    return full_prompt_text, batch_mean, batch_std

def create_descriptive_task_prompt():
    return (
        "Task: Estimate the underlying Normal distribution parameters for the Target Item.\n\n"
        "STRICT OUTPUT FORMAT:\n"
        "You must start your response with exactly this line:\n"
        "PREDICTION -> Mean: [Value], Std: [Value]\n\n"
        "Then, on a new line, provide your:\n"
        "REASONING -> [Your detailed explanation here]"
    )

# ==========================================
# API EXECUTION
# ==========================================
def call_gemini_api(prompt):
    try:
        # Define the configuration (Temperature, System Prompt, Safety)
        # In the new SDK, system instructions and safety settings go into 'config'
        config = types.GenerateContentConfig(
            temperature=0.0,
            system_instruction="You are a helpful forecasting assistant.",
            safety_settings=[
                types.SafetySetting(
                    category="HARM_CATEGORY_HATE_SPEECH",
                    threshold="BLOCK_NONE"
                ),
                types.SafetySetting(
                    category="HARM_CATEGORY_DANGEROUS_CONTENT",
                    threshold="BLOCK_NONE"
                ),
                types.SafetySetting(
                    category="HARM_CATEGORY_HARASSMENT",
                    threshold="BLOCK_NONE"
                ),
                types.SafetySetting(
                    category="HARM_CATEGORY_SEXUALLY_EXPLICIT",
                    threshold="BLOCK_NONE"
                )
            ]
        )

        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=config
        )

        return response.text.strip()
    except Exception as e:
        print(f"\n[ERROR] API Call Failed: {e}")
        return None

def extract_metrics(text):
    pred_mean, pred_std = None, None
    if not text: return None, None

    match_line = re.search(r"PREDICTION ->(.*?)(?:\n|$)", text, re.IGNORECASE)

    if match_line:
        prediction_text = match_line.group(1)
        mean_match = re.search(r"Mean[:\s=]+([\d\.]+)", prediction_text, re.IGNORECASE)
        std_match = re.search(r"(?:Std|Deviation)[:\s=]+([\d\.]+)", prediction_text, re.IGNORECASE)

        if mean_match:
            try: pred_mean = float(mean_match.group(1))
            except: pass
        if std_match:
            try: pred_std = float(std_match.group(1))
            except: pass

    return pred_mean, pred_std

# ==========================================
# MAIN LOOP
# ==========================================
if __name__ == "__main__":
    if not os.path.exists(INPUT_DATA_FILE):
        print(f"Error: {INPUT_DATA_FILE} not found.")
    else:
        df = load_and_prep_data(INPUT_DATA_FILE)
        results = []

        total_items = len(df)
        print(f"\n[START] Processing ALL {total_items} items (Gemini New SDK Strategy)...")

        for idx in range(total_items):
            target_row = df.iloc[idx]
            article_id = target_row['article_id']

            # 1. Generate Prompt
            context_prompt, context_batch_mean, context_batch_std = create_full_context_prompt(target_row, df)
            full_prompt = f"{context_prompt}\n{create_descriptive_task_prompt()}"

            sys.stdout.write(f"\rItem {idx+1}/{total_items} (ID: {article_id})...")
            sys.stdout.flush()

            # 2. Call API
            output = call_gemini_api(full_prompt)

            if output:
                pred_mean, pred_std = extract_metrics(output)

                true_mean = target_row['mean_sales']
                if pred_mean is not None:
                    error_bias = pred_mean - true_mean
                    abs_error = abs(error_bias)
                else:
                    error_bias = None
                    abs_error = None

                results.append({
                    "article_id": article_id,
                    "prod_name": target_row['prod_name'],

                    # Truth
                    "true_mean": true_mean,
                    "true_std": target_row['std_sales'],

                    # Prediction
                    "pred_mean": pred_mean,
                    "pred_std": pred_std,
                    "error_bias": error_bias,
                    "abs_error": abs_error,

                    # Context Stats
                    "context_batch_mean": context_batch_mean,
                    "context_batch_std": context_batch_std,

                    "gemini_full_response": output
                })

            # Rate Limiting: Sleep briefly to avoid hitting RPM limits
            time.sleep(1.5)

            # Save incrementally
            if (idx + 1) % 10 == 0:
                pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)

        # FINAL SAVE
        if results:
            pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)
            print(f"\n\n[DONE] Results saved to: {OUTPUT_FILE}")

#### mistral

In [ ]:
import pandas as pd
import numpy as np
import requests
import sys
import os
import re
import time
import json

# ==========================================
# CONFIGURATION
# ==========================================
# [SECURITY NOTE] Replaced key with placeholder for safety.
# Please insert your actual key back here.
API_KEY = "..."

# Mistral API Configuration
API_URL = "https://api.mistral.ai/v1/chat/completions"
MODEL_NAME = "mistral-large-latest"

# --- FILE NAMES ---
INPUT_DATA_FILE = 'Trousers_new.csv'
OUTPUT_FILE     = 'mistral_predictions_descriptive.csv'
# ------------------

# ==========================================
# DATA LOADING
# ==========================================
def load_and_prep_data(filepath):
    print(f"[INIT] Loading data from {filepath}...")
    df = pd.read_csv(filepath)
    sales_cols = [c for c in df.columns if 'count' in c]

    if not sales_cols:
        print("[WARNING] No sales columns found containing 'count'.")

    df['mean_sales'] = df[sales_cols].mean(axis=1)
    df['std_sales'] = df[sales_cols].std(axis=1)
    return df

# ==========================================
# PROMPT GENERATORS
# ==========================================
def get_features_text(row):
    features = [
        'prod_name', 'product_type_name', 'graphical_appearance_name',
        'colour_group_name', 'department_name', 'index_name',
        'section_name', 'garment_group_name', 'detail_desc'
    ]
    feature_text = ""
    for feat in features:
        if feat in row.index:
            val = row[feat]
            feature_text += f"  - {feat}: {val}\n"
    return feature_text

def create_full_context_prompt(target_row, df):
    # 1. Target Item Task
    target_features = get_features_text(target_row)
    base_target_text = (
        f"Target Item Task:\n"
        f"Please estimate the demand distribution for:\n"
        f"- Product Name: {target_row['prod_name']}\n"
        f"- Features:\n{target_features}"
    )

    # 2. Context: Randomly sample 100 items (excluding target)
    other_items = df[df.index != target_row.name]

    if len(other_items) > 100:
        other_items = other_items.sample(n=100)

    # --- Calculate Stats for this specific batch ---
    batch_mean = other_items['mean_sales'].mean()
    batch_std  = other_items['std_sales'].mean()
    # -----------------------------------------------

    examples_str = f"Context ({len(other_items)} Reference Items with known Demand Statistics):\n"

    for i, (_, row) in enumerate(other_items.iterrows(), 1):
        examples_str += f"\n[Reference Item {i}]\n"
        examples_str += f"Features:\n{get_features_text(row)}"
        examples_str += f"KNOWN STATISTICS -> Mean: {row['mean_sales']:.2f}, Std: {row['std_sales']:.2f}\n"
        examples_str += "-" * 30

    intro = "You are an expert in inventory demand forecasting. Use the provided reference items to estimate statistics for the Target Item.\n\n"

    full_prompt_text = intro + examples_str + "\n\n" + base_target_text

    return full_prompt_text, batch_mean, batch_std

def create_descriptive_task_prompt():
    return (
        "Task: Estimate the underlying Normal distribution parameters for the Target Item.\n\n"
        "STRICT OUTPUT FORMAT:\n"
        "You must start your response with exactly this line:\n"
        "PREDICTION -> Mean: [Value], Std: [Value]\n\n"
        "Then, on a new line, provide your:\n"
        "REASONING -> [Your detailed explanation here]"
    )

# ==========================================
# API EXECUTION (Robust with Retries)
# ==========================================
def call_mistral_api(prompt, retries=3):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
        "Accept": "application/json"
    }

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": "You are a helpful forecasting assistant."},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.0
    }

    for attempt in range(retries):
        try:
            response = requests.post(API_URL, headers=headers, json=payload, timeout=60)

            if response.status_code == 200:
                data = response.json()
                if "choices" in data and len(data["choices"]) > 0:
                    return data["choices"][0]["message"]["content"].strip()
            elif response.status_code == 429:
                # Rate limit hit
                print(f"[WARN] Rate limited. Waiting 10s... (Attempt {attempt+1}/{retries})")
                time.sleep(10)
            else:
                print(f"[ERROR] HTTP {response.status_code}: {response.text}")

        except Exception as e:
            print(f"[ERROR] Connection failed: {e}")

        # Exponential backoff before retry (2s, 4s, 8s...)
        time.sleep(2 ** (attempt + 1))

    return None

def extract_metrics(text):
    pred_mean, pred_std = None, None
    if not text: return None, None

    match_line = re.search(r"PREDICTION ->(.*?)(?:\n|$)", text, re.IGNORECASE)

    if match_line:
        prediction_text = match_line.group(1)
        mean_match = re.search(r"Mean[:\s=]+([\d\.]+)", prediction_text, re.IGNORECASE)
        std_match = re.search(r"(?:Std|Deviation)[:\s=]+([\d\.]+)", prediction_text, re.IGNORECASE)

        if mean_match:
            try: pred_mean = float(mean_match.group(1))
            except: pass
        if std_match:
            try: pred_std = float(std_match.group(1))
            except: pass

    return pred_mean, pred_std

# ==========================================
# MAIN LOOP
# ==========================================
if __name__ == "__main__":
    if not os.path.exists(INPUT_DATA_FILE):
        print(f"Error: {INPUT_DATA_FILE} not found.")
    else:
        df = load_and_prep_data(INPUT_DATA_FILE)
        results = []

        total_items = len(df)
        print(f"\n[START] Processing ALL {total_items} items (Mistral Large Direct HTTP)...")

        for idx in range(total_items):
            target_row = df.iloc[idx]
            article_id = target_row['article_id']

            # 1. Generate Prompt
            context_prompt, context_batch_mean, context_batch_std = create_full_context_prompt(target_row, df)
            full_prompt = f"{context_prompt}\n{create_descriptive_task_prompt()}"

            sys.stdout.write(f"\rItem {idx+1}/{total_items} (ID: {article_id})...")
            sys.stdout.flush()

            # 2. Call API
            output = call_mistral_api(full_prompt)

            if output:
                pred_mean, pred_std = extract_metrics(output)

                true_mean = target_row['mean_sales']
                if pred_mean is not None:
                    error_bias = pred_mean - true_mean
                    abs_error = abs(error_bias)
                else:
                    error_bias = None
                    abs_error = None

                results.append({
                    "article_id": article_id,
                    "prod_name": target_row['prod_name'],

                    # Truth
                    "true_mean": true_mean,
                    "true_std": target_row['std_sales'],

                    # Prediction
                    "pred_mean": pred_mean,
                    "pred_std": pred_std,
                    "error_bias": error_bias,
                    "abs_error": abs_error,

                    # Context Stats
                    "context_batch_mean": context_batch_mean,
                    "context_batch_std": context_batch_std,

                    "mistral_full_response": output
                })

            # Rate Limiting: 2.5 seconds safe buffer
            time.sleep(2.5)

            # Save incrementally
            if (idx + 1) % 10 == 0:
                pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)

        # FINAL SAVE
        if results:
            pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)
            print(f"\n\n[DONE] Results saved to: {OUTPUT_FILE}")